# Our data

In [1]:
import numpy as np
import os
import librosa
import re

# Define the dataset path
data_path = "/mnt/d/PhD/ALS_Diagnosis_Meta/Dataset/FINAL_Balanced_CORRECTED"
# Define class labels
LABELS = {
    "Control": 0,
    "ALSwDysarthria": 1,
    "ALSwoDysarthria": 2
}

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text for text in re.split(r'(\d+)', s)]

def read_data(base_path):
    s_data = []
    s_label = []
    s_sr = []
    file_list = []  # Store file identifiers

    # Iterate through all categories (Control, ALSwDysarthria, ALSwoDysarthria)
    for category, label in LABELS.items():
        category_path = os.path.join(base_path, category)
        
        # Get the subject folders with natural sorting
        subjects = sorted(
            [f for f in os.listdir(category_path) if os.path.isdir(os.path.join(category_path, f))],
            key=natural_sort_key
)

        for subject in subjects:
            subject_path = os.path.join(category_path, subject)

            print(f"\nProcessing Subject: {subject} ({category})")

            # Sort files naturally
            wav_files = sorted([f for f in os.listdir(subject_path) if f.endswith('.wav') or f.endswith('.WAV')], key=natural_sort_key)

            # Process each WAV file separately
            for wav_file in wav_files:
                file_path = os.path.join(subject_path, wav_file)
                
                # Load using librosa instead of torchaudio
                try:
                    waveform, sample_rate = librosa.load(file_path, sr=None)
                    # print(f"✅ Processed {wav_file} for {subject}, Shape: {waveform.shape}, Sample rate: {sample_rate}")
                    
                    # Convert to mono if necessary (librosa should already give mono if file is single channel)
                    if len(waveform.shape) > 1 and waveform.shape[0] > 1:
                        waveform = waveform.mean(axis=0)
                    
                    # Store each file separately
                    s_data.append(waveform)
                    s_label.append(label)
                    s_sr.append(sample_rate)
                    # IMPORTANT: include category to avoid collapsing subject IDs like F1 across groups
                    file_list.append(f"{category}/{subject}/{wav_file}")

                except Exception as e:
                    print(f"❌ Error loading file {wav_file}: {e}")

    return s_data, s_label, s_sr, file_list

# Load the dataset
s_data, s_label, s_sr, file_list = read_data(data_path)




Processing Subject: F1_denoised (Control)

Processing Subject: F2_denoised (Control)

Processing Subject: F3_denoised (Control)

Processing Subject: F4_denoised (Control)

Processing Subject: F5_denoised (Control)

Processing Subject: F6_denoised (Control)

Processing Subject: F7_denoised (Control)

Processing Subject: F8_denoised (Control)

Processing Subject: F9_denoised (Control)

Processing Subject: F10_denoised (Control)

Processing Subject: M1_denoised (Control)

Processing Subject: M2_denoised (Control)

Processing Subject: M3_denoised (Control)

Processing Subject: M4_denoised (Control)

Processing Subject: M5_denoised (Control)

Processing Subject: M6_denoised (Control)

Processing Subject: M7_denoised (Control)

Processing Subject: M8_denoised (Control)

Processing Subject: M9_denoised (Control)

Processing Subject: M10_denoised (Control)

Processing Subject: F1_denoised (ALSwDysarthria)

Processing Subject: F2_denoised (ALSwDysarthria)

Processing Subject: F3_denoised (ALSw

In [2]:
import torch
import torchaudio
from transformers import WavLMModel
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

# Reproducibility (does not change embeddings in eval/no_grad, but makes runs consistent)
import random
import sys
import platform
import transformers
import librosa

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXTRACTION_ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": getattr(np, "__version__", "unknown"),
    "torch": getattr(torch, "__version__", "unknown"),
    "torchaudio": getattr(torchaudio, "__version__", "unknown"),
    "transformers": getattr(transformers, "__version__", "unknown"),
    "librosa": getattr(librosa, "__version__", "unknown"),
}

# ---------------------------
# 1. Resample and Preprocess Data
# ---------------------------
def resample_to_16khz(s_data, s_sr, target_freq=16000):
    """Resample each waveform to 16kHz using its true original sample rate."""
    resampled = []
    for waveform, orig_freq in zip(s_data, s_sr):
        # Convert each waveform (assumed to be a NumPy array) to a torch.Tensor (float32)
        wav_tensor = torch.tensor(waveform, dtype=torch.float32)
        if wav_tensor.dim() != 1:
            wav_tensor = wav_tensor.squeeze()
        if int(orig_freq) == int(target_freq):
            resampled.append(wav_tensor)
            continue
        resampler = torchaudio.transforms.Resample(orig_freq=int(orig_freq), new_freq=int(target_freq))
        resampled.append(resampler(wav_tensor))
    return resampled

# Protocol info to persist alongside embeddings
PREPROCESSING_PROTOCOL = {
    "input_loading": {
        "library": "librosa",
        "sr": None,
        "mono": True,
    },
    "resampling": {
        "target_sample_rate_hz": 16000,
        "method": "torchaudio.transforms.Resample",
        "orig_sample_rate_hz": "per-file (read from librosa.load(sr=None))",
        "note": "If all WAVs are 44100Hz, this matches orig_freq=44100 behavior.",
    },
    "segmentation": {
        "trimming": "none",
        "max_length_policy": "no truncation",
    },
    "batching": {
        "padding": "pad_sequence(batch_first=True, padding_value=0)",
        "attention_mask": "1 for valid samples, 0 for padded",
    },
}

# data is a list of tensors with shape [num_samples]
data = resample_to_16khz(s_data, s_sr)

# Convert each tensor to a 1D tensor and wrap in a dict with key "input_values"
preprocessed_data = []
for waveform in data:
    # waveform shape is [num_samples]
    waveform = waveform.squeeze()
    preprocessed_data.append({"input_values": waveform})

# 2. Define Collate Function for Variable-Length Inputs
# ---------------------------
def collate_fn(batch):
    """
    Collate function to pad variable-length "input_values" and create an attention mask.
    Each item in `batch` is a dict with key "input_values" (a 1D tensor).
    """
    input_values_list = []
    for item in batch:
        # Expecting item is a dict with "input_values" as a tensor.
        tensor = item["input_values"]
        # Ensure tensor is a 1D tensor.
        if tensor.dim() != 1:
            tensor = tensor.squeeze()
        input_values_list.append(tensor)
    
    # Pad the sequences: resulting shape [batch_size, max_seq_length]
    padded_inputs = pad_sequence(input_values_list, batch_first=True, padding_value=0)
    
    # Create attention masks based on original lengths
    attention_masks = [torch.ones(tensor.size(0), dtype=torch.long) for tensor in input_values_list]
    padded_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    
    return {"input_values": padded_inputs, "attention_mask": padded_masks}

BATCH_SIZE = 8
data_loader = DataLoader(preprocessed_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)
DATA_LOADER_CONFIG = {"batch_size": BATCH_SIZE, "shuffle": False}

for batch in data_loader:
    print(f"Input 'input_values' shape: {batch['input_values'].shape}")
    print(f"Input 'attention_mask' shape: {batch['attention_mask'].shape}")
    break 

Input 'input_values' shape: torch.Size([8, 10968])
Input 'attention_mask' shape: torch.Size([8, 10968])


In [3]:
from transformers import WavLMModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = WavLMModel.from_pretrained("microsoft/wavlm-large")
model.to(device)
model.eval()

# Explicit: frozen model for feature extraction (no fine-tuning here)
for p in model.parameters():
    p.requires_grad = False

print(f"✅ Using frozen WavLM for embedding extraction only. Device={device}")
print(model)


✅ Using frozen WavLM for embedding extraction only. Device=cuda
WavLMModel(
  (feature_extractor): WavLMFeatureEncoder(
    (conv_layers): ModuleList(
      (0): WavLMLayerNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (1-4): 4 x WavLMLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (5-6): 2 x WavLMLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): WavLMFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
   

In [4]:
import numpy as np
import torch
import time

# Define layer configurations
LAYER_CONFIGS = {
    "stability": [5, 6, 7],        # ≈ Hubert [9,10,11] (mid-depth)
    "acoustic": [0, 1, 2],        # ≈ Hubert [0,1,2] (shallow)
    "middle": [10, 11, 12],    # ≈ Hubert [20,21,22] (disease-relevant)
    "last3": [22, 23, 24],        # deep layers (last 3)
    "multi-depth": [0, 5, 12, 24],
    "all": list(range(0, 25)),
}

EXTRACTION_STATS = {}

def extract_and_save_features(layer_sets=LAYER_CONFIGS):
    """
    Returns dictionary of features (extraction-only; model is frozen)
    """
    global EXTRACTION_STATS
    # Dictionary to store features
    all_features = {name: [] for name in layer_sets}
    batch_times_sec = []
    batch_sizes = []
    total_utts = 0

    t_total_start = time.perf_counter()
    for batch in data_loader:
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        bs = int(input_values.shape[0])
        total_utts += bs

        t0 = time.perf_counter()
        # Single forward pass
        with torch.no_grad():
            outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)

        # Process each layer configuration
        for name, layers in layer_sets.items():
            pooled_features = []
            for layer in layers:
                feature = outputs.hidden_states[layer]
                seq_len = feature.size(1)
                adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()

                # Masked mean pooling
                masked_feature = feature * adj_mask
                valid_counts = adj_mask.sum(dim=1).clamp(min=1e-9)
                mean_pooled = masked_feature.sum(dim=1) / valid_counts
                pooled_features.append(mean_pooled)

            # Concatenate and store
            final_features = torch.cat(pooled_features, dim=-1).cpu().numpy()
            all_features[name].append(final_features)
        t1 = time.perf_counter()

        batch_times_sec.append(t1 - t0)
        batch_sizes.append(bs)

    # Concatenate all batches for each feature set
    for name in all_features:
        all_features[name] = np.concatenate(all_features[name], axis=0)

    t_total_end = time.perf_counter()
    total_sec = t_total_end - t_total_start
    EXTRACTION_STATS = {
        "total_utterances": int(total_utts),
        "batch_sizes": batch_sizes,
        "batch_times_sec": batch_times_sec,
        "total_time_sec": float(total_sec),
        "mean_time_sec_per_utt": float(total_sec / max(total_utts, 1)),
        "device": str(device),
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

    return all_features

# Now this will work correctly
all_features = extract_and_save_features()


/home/ali_shah/anaconda3/envs/ali_torch/lib/python3.13/site-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [5]:
last3 = all_features["last3"]           # shape: [n_samples, 3840]
middle3 = all_features["middle"]       # shape: [n_samples, 3840]  
acoustic = all_features["acoustic"]     # shape: [n_samples, 3840]
stability = all_features["stability"]   # shape: [n_samples, 3840]
full_spec = all_features["multi-depth"]
all_layers = all_features["all"]

# Save EVERYTHING (embeddings + metadata) into Embeddings/ in a shareable, GitHub-friendly folder
import json
from pathlib import Path

# Notebook lives in code/, so this resolves to <repo>/Embeddings/...
out_dir = Path("..") / "Embeddings" / "wavlm_large_mydata"
out_dir.mkdir(parents=True, exist_ok=True)

# Embeddings (same filenames as before, but now stored under Embeddings/wavlm_large_mydata/)
np.save(out_dir / "xwavlm_last3.npy", last3)
np.save(out_dir / "xwavlm_middle3.npy", middle3)
np.save(out_dir / "xwavlm_acoustic.npy", acoustic)
np.save(out_dir / "xwavlm_stability.npy", stability)
np.save(out_dir / "xwavlm_fullspec.npy", full_spec)
np.save(out_dir / "xwavlm_all.npy", all_layers)
np.save(out_dir / "labels.npy", np.asarray(s_label))

# Per-utterance manifest (required to prove subject-wise splits later)
np.save(out_dir / "file_list.npy", np.asarray(file_list, dtype=object))

def _subject_id_from_file_id(file_id: object) -> str:
    parts = str(file_id).split('/')
    # Expected: category/subject/filename.wav
    return '/'.join(parts[:2]) if len(parts) >= 2 else parts[0]

subject_ids = np.asarray([_subject_id_from_file_id(f) for f in file_list], dtype=object)
np.save(out_dir / "subject_ids.npy", subject_ids)

# Save the label map safely (dict, not just keys)
np.save(out_dir / "label_map.npy", LABELS, allow_pickle=True)

# Compute/cost logging (Reviewer 2 #6)
try:
    (out_dir / "compute_log.json").write_text(json.dumps(EXTRACTION_STATS, indent=2))
except Exception as e:
    print(f"⚠️ Could not write compute_log.json: {e}")

# Extraction config for reproducibility (Reviewer 2 #1/#3)
config = {
    "data_path": str(data_path),
    "model_name": "microsoft/wavlm-large",
    "model_frozen": True,
    "seed": globals().get("SEED", None),
    "env": globals().get("EXTRACTION_ENV", {}),
    "preprocessing_protocol": globals().get("PREPROCESSING_PROTOCOL", {}),
    "data_loader": globals().get("DATA_LOADER_CONFIG", {}),
    "pooling": "masked_mean (attention-mask aware)",
    "layer_configs": LAYER_CONFIGS,
}
(out_dir / "extraction_config.json").write_text(json.dumps(config, indent=2))

print(f"✅ Saved WavLM export to: {out_dir.resolve()}")

# Print statements remain identical
print(f"last3     shape: {last3.shape}")
print(f"middle3   shape: {middle3.shape}")
print(f"acoustic  shape: {acoustic.shape}")
print(f"stability shape: {stability.shape}")
print(f"full_spec shape: {full_spec.shape}")


✅ Saved WavLM export to: /mnt/d/PhD/ALS_Diagnosis_Meta/Embeddings/wavlm_large_mydata
last3     shape: (6200, 3072)
middle3   shape: (6200, 3072)
acoustic  shape: (6200, 3072)
stability shape: (6200, 3072)
full_spec shape: (6200, 4096)
